In [1]:
from sklearn.datasets import fetch_openml
import pandas as pd

data = fetch_openml('credit-g', version=1, as_frame=True)
X, y = data.data, data.target

print(f"Shape: {X.shape}")
print(f"\nClass counts:\n{y.value_counts()}")
print(f"\nClass proportions:\n{y.value_counts(normalize=True)}")

Shape: (1000, 20)

Class counts:
class
good    700
bad     300
Name: count, dtype: int64

Class proportions:
class
good    0.7
bad     0.3
Name: proportion, dtype: float64


In [2]:
num_cols = X.select_dtypes(include='number').columns.tolist()
cat_cols = X.select_dtypes(include=['category', 'object']).columns.tolist()

print(f"Numeric ({len(num_cols)}):\n{num_cols}\n")
print(f"Categorical ({len(cat_cols)}):\n{cat_cols}\n")
print(f"Missing values total: {X.isnull().sum().sum()}")

Numeric (7):
['duration', 'credit_amount', 'installment_commitment', 'residence_since', 'age', 'existing_credits', 'num_dependents']

Categorical (13):
['checking_status', 'credit_history', 'purpose', 'savings_status', 'employment', 'personal_status', 'other_parties', 'property_magnitude', 'other_payment_plans', 'housing', 'job', 'own_telephone', 'foreign_worker']

Missing values total: 0


In [3]:
banned = ['personal_status', 'foreign_worker']

X_full = X.copy()
X_clean = X.drop(columns=banned)

cat_cols_clean = [c for c in cat_cols if c not in banned]

print(f"Original: {X_full.shape[1]} columns")
print(f"Clean:    {X_clean.shape[1]} columns")
print(f"Removed:  {banned}")

Original: 20 columns
Clean:    18 columns
Removed:  ['personal_status', 'foreign_worker']


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(f"Train: {X_train.shape[0]} rows")
print(f"Test:  {X_test.shape[0]} rows")
print(f"\nTrain mix:\n{y_train.value_counts(normalize=True)}")
print(f"Test mix:\n{y_test.value_counts(normalize=True)}")

Train: 800 rows
Test:  200 rows

Train mix:
class
good    0.7
bad     0.3
Name: proportion, dtype: float64
Test mix:
class
good    0.7
bad     0.3
Name: proportion, dtype: float64


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_cols_clean = [c for c in num_cols if c not in banned]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols_clean),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols_clean)
    ]
)

print(f"Numeric columns to scale: {len(num_cols_clean)}")
print(f"Word columns to encode:   {len(cat_cols_clean)}")

Numeric columns to scale: 7
Word columns to encode:   11


In [6]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

logreg = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

logreg.fit(X_train, y_train)

print("Model trained.")
print(f"Learned from {X_train.shape[0]} people.")

Model trained.
Learned from 800 people.


In [7]:
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

y_pred = logreg.predict(X_test)
y_proba = logreg.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba, labels=['bad', 'good'])

print(f"Accuracy: {acc:.3f}   (baseline to beat: 0.700)")
print(f"AUC:      {auc:.3f}")
print()
print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred))

Accuracy: 0.700   (baseline to beat: 0.700)
AUC:      0.747

[[ 30  30]
 [ 30 110]]

              precision    recall  f1-score   support

         bad       0.50      0.50      0.50        60
        good       0.79      0.79      0.79       140

    accuracy                           0.70       200
   macro avg       0.64      0.64      0.64       200
weighted avg       0.70      0.70      0.70       200



In [8]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb = Pipeline(steps=[
    ('prep', preprocessor),
    ('model', HistGradientBoostingClassifier(random_state=42))
])

gb.fit(X_train, y_train)

gb_pred = gb.predict(X_test)
gb_proba = gb.predict_proba(X_test)[:, 1]

gb_acc = accuracy_score(y_test, gb_pred)
gb_auc = roc_auc_score(y_test, gb_proba, labels=['bad', 'good'])

print(f"{'Model':<22}{'Accuracy':<12}{'AUC'}")
print(f"{'Logistic Regression':<22}{acc:<12.3f}{auc:.3f}")
print(f"{'Gradient Boosting':<22}{gb_acc:<12.3f}{gb_auc:.3f}")
print()
print(confusion_matrix(y_test, gb_pred))

Model                 Accuracy    AUC
Logistic Regression   0.700       0.747
Gradient Boosting     0.740       0.772

[[ 31  29]
 [ 23 117]]
